# create3d: image / text → 3D model (Hunyuan3D-2)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sajjadnisham/3d/blob/main/create3d_colab.ipynb)

Turn a photo or a text prompt into a 3D model with [Tencent Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2),
and download it as **GLB, OBJ, FBX, STL or PLY**.

**Before you start:** *Runtime → Change runtime type → **T4 GPU*** (or better), then run the cells top to bottom.

| Step | Time on a free T4 |
|---|---|
| 1. Install (shape only) | ~3 min |
| 1. Install with texture | ~8–12 min (compiles CUDA code) |
| 3. First generation | + model download (~5–10 GB, once per session) |

Hunyuan3D-2 is under the [Tencent Hunyuan Non-Commercial License](https://github.com/Tencent-Hunyuan/Hunyuan3D-2/blob/main/LICENSE).

In [ ]:
#@title 1. Install { display-mode: "form" }
#@markdown Tick this to be able to paint textures (colours) onto models. It adds a few minutes of compiling.
ENABLE_TEXTURE = True #@param {type:"boolean"}

import os, site, subprocess, sys, importlib

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then run this cell again.")
print("GPU:", gpu.stdout.strip())

REPO = "/content/3d"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/sajjadnisham/3d {REPO}
else:
    !git -C {REPO} pull -q

!apt-get -qq install -y assimp-utils > /dev/null   # for FBX export
print("Installing Hunyuan3D-2 + create3d ...")
cmd = ["bash", "scripts/install.sh"] + (["--texture"] if ENABLE_TEXTURE else [])
res = subprocess.run(cmd, cwd=REPO, capture_output=True, text=True)
if res.returncode != 0:
    print((res.stdout + res.stderr)[-4000:])
    raise SystemExit("Install failed, see the log above.")

# Pinned versions may replace packages this runtime already loaded (e.g. numpy, Pillow).
from importlib.metadata import version
stale = [d for m, d in (("numpy", "numpy"), ("PIL", "pillow")) if m in sys.modules
         and getattr(sys.modules[m], "__version__", None) != version(d)]
if stale:
    raise SystemExit(f"Updated {', '.join(stale)}: click Runtime > Restart session, then run this cell "
                     "again (it is quick the second time).")

# Make the freshly installed packages importable without restarting the runtime.
for p in site.getsitepackages():
    site.addsitedir(p)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
importlib.invalidate_caches()

import torch, hy3dgen, create3d
if ENABLE_TEXTURE:
    import custom_rasterizer
print("✅ Install finished")

In [ ]:
#@title 2. Load the model { display-mode: "form" }
#@markdown `mini-turbo` is the fastest; `full` gives the most detail; `mv-turbo` is for several photos of one object.
SHAPE_MODEL = "mini-turbo" #@param ["mini-turbo", "mini", "turbo", "full", "mv-turbo", "mv"]
TEXTURE_MODEL = "turbo" #@param ["turbo", "full"]
#@markdown Keep this on for texture on a free T4 (15 GB): it moves idle models to CPU memory.
LOW_VRAM = True #@param {type:"boolean"}

from create3d.engine import Generator3D, GenerationSettings, export_mesh

gen = Generator3D(shape_model=SHAPE_MODEL, texture_model=TEXTURE_MODEL, device="cuda", low_vram=LOW_VRAM)
print(f"✅ Ready: {SHAPE_MODEL} (weights download on the first generation)")

In [ ]:
#@title 3. Choose your input { display-mode: "form" }
#@markdown **Upload image**: one object, centred, plain background works best.<br>
#@markdown **Multi-view**: upload files named `front`, `left`, `back`, `right` (needs `mv`/`mv-turbo` in step 2).
INPUT = "Upload image" #@param ["Upload image", "Demo image", "Text prompt", "Multi-view upload"]
PROMPT = "a cute cartoon robot, toy" #@param {type:"string"}

import shutil
from pathlib import Path
from google.colab import files
from PIL import Image

IN_DIR = Path("/content/inputs"); IN_DIR.mkdir(exist_ok=True)
source = {}

if INPUT == "Upload image":
    up = files.upload()
    name = next(iter(up))
    source = {"image": str(IN_DIR / name)}
    shutil.move(name, source["image"])
    NAME = Path(name).stem
elif INPUT == "Demo image":
    demo = IN_DIR / "demo.png"
    if not demo.exists():
        !wget -q -O {demo} https://raw.githubusercontent.com/Tencent-Hunyuan/Hunyuan3D-2/f8db63096c8282cb27354314d896feba5ba6ff8a/assets/demo.png
    source = {"image": str(demo)}
    NAME = "demo"
elif INPUT == "Text prompt":
    source = {"prompt": PROMPT}
    NAME = "".join(c if c.isalnum() else "_" for c in PROMPT)[:40].strip("_") or "model"
else:
    if not gen.is_multiview:
        raise SystemExit("Pick mv-turbo or mv in step 2 for multi-view.")
    up = files.upload()
    views = {}
    for name in up:
        key = Path(name).stem.lower()
        if key not in ("front", "left", "back", "right"):
            print(f"Skipping {name}: name it front/left/back/right"); continue
        dest = IN_DIR / name; shutil.move(name, dest); views[key] = str(dest)
    if not views:
        raise SystemExit("No usable views uploaded.")
    source = {"views": views}
    NAME = "multiview"

if "image" in source:
    display(Image.open(source["image"]).resize((256, 256)))
print("Input:", INPUT, "→", NAME)

In [ ]:
#@title 4. Generate the 3D model { display-mode: "form" }
TEXTURE = True #@param {type:"boolean"}
#@markdown Higher octree resolution = finer detail, more time and memory.
OCTREE_RESOLUTION = 256 #@param [192, 256, 320, 384] {type:"raw"}
#@markdown 0 = model default (5 for turbo, 50 for others).
STEPS = 0 #@param {type:"integer"}
SEED = 1234 #@param {type:"integer"}
#@markdown 0 = keep every face.
MAX_FACES = 40000 #@param {type:"integer"}
REMOVE_BACKGROUND = True #@param {type:"boolean"}
#@markdown Formats to save:
GLB = True #@param {type:"boolean"}
OBJ = False #@param {type:"boolean"}
FBX = True #@param {type:"boolean"}
STL = False #@param {type:"boolean"}
PLY = False #@param {type:"boolean"}

import time, torch

if TEXTURE and not ENABLE_TEXTURE:
    raise SystemExit("Texture needs ENABLE_TEXTURE ticked in step 1 (re-run step 1).")

settings = GenerationSettings(
    steps=STEPS or None,
    octree_resolution=int(OCTREE_RESOLUTION),
    seed=int(SEED),
    max_faces=MAX_FACES or None,
    texture=TEXTURE,
)
start = time.time()
mesh, reference = gen.generate(settings=settings, remove_background=REMOVE_BACKGROUND, **source)
print(f"Generated in {time.time() - start:.0f}s: {len(mesh.vertices):,} vertices, {len(mesh.faces):,} faces")

OUT_DIR = Path("/content/outputs"); OUT_DIR.mkdir(exist_ok=True)
stem = f"{NAME}_{SEED}"
reference.save(OUT_DIR / f"{stem}_input.png")
saved = []
preview_glb = export_mesh(mesh, OUT_DIR / f"{stem}.glb")   # always made, used for the preview
if GLB:
    saved.append(preview_glb)
for ext, on in ((".obj", OBJ), (".fbx", FBX), (".stl", STL), (".ply", PLY)):
    if on:
        saved.append(export_mesh(mesh, OUT_DIR / f"{stem}{ext}"))
torch.cuda.empty_cache()
display(reference.resize((256, 256)))

In [ ]:
#@title 5. Preview (drag to rotate, scroll to zoom) { display-mode: "form" }
import base64
from IPython.display import HTML

b64 = base64.b64encode(preview_glb.read_bytes()).decode()
HTML(f"""
<script type="module" src="https://ajax.googleapis.com/ajax/libs/model-viewer/3.5.0/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{b64}" camera-controls auto-rotate shadow-intensity="1"
  exposure="1" environment-image="neutral" style="width:100%;height:520px;background:#eee;border-radius:8px">
</model-viewer>
""")

In [ ]:
#@title 6. Download { display-mode: "form" }
#@markdown Downloads every format you ticked in step 4. Tick to also copy them to Google Drive (MyDrive/create3d).
SAVE_TO_DRIVE = False #@param {type:"boolean"}

for f in saved:
    print("⬇️", f.name)
    files.download(str(f))

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = Path("/content/drive/MyDrive/create3d"); dest.mkdir(parents=True, exist_ok=True)
    for f in saved + [OUT_DIR / f"{stem}_input.png"]:
        shutil.copy(f, dest)
    print("Copied to", dest)

### Make another one
Go back to **step 3** (new input) or **step 4** (new seed / settings). Steps 1–2 only need to run once per session.

### Optional: browser interface
The cell below opens the create3d web app with a public link, reusing the model already loaded.
It keeps running until you stop the cell.

In [ ]:
#@title 7. Web interface (optional) { display-mode: "form" }
from create3d.app import build_ui

build_ui(gen).queue(max_size=8).launch(share=True, debug=True)

### Troubleshooting
- **CUDA out of memory**: keep *LOW_VRAM* on, lower the octree resolution, or untick texture. *Runtime → Restart session* clears memory (then run steps 1–2 again; installation is quick the second time).
- **Holes or blobs**: try a different *SEED*, a higher octree resolution, or the `full` model.
- **Background left in the model**: upload a PNG with a transparent background.
- **Text prompt runs out of memory**: text → 3D also loads HunyuanDiT; untick texture, or use a bigger GPU (L4 / A100).